# Notebook 01: Dataset Understanding

## Introduction
As the initial phase of our data science workflow, this notebook is dedicated to conducting a detailed inspection of the raw data files provided for the Walmart Store Sales Forecasting project. Understanding the raw schema, identifying the target variables, checking data shapes, and spotting immediate data quality issues are essential steps before we design our cleaning and preprocessing pipeline.

## Dataset Files Available:
1. **`train.csv`**: Historical training data containing weekly sales, store, department, date, and holiday flags from February 2010 to October 2012.
2. **`test.csv`**: The test set for which we need to predict weekly sales. The date range spans from November 2012 to July 2013.
3. **`stores.csv`**: Store-level metadata containing information about store type (A, B, or C) and its size.
4. **`features.csv`**: Contains additional environmental and macroeconomic indicators (e.g., temperature, fuel price, CPI, unemployment) corresponding to stores and dates.

In [1]:
import sys
import os
import pandas as pd

# Add the project root to python path to import our modules
sys.path.append(os.path.abspath('..'))

## 1. Load Raw Datasets
Here, we import the raw files into Pandas DataFrames using our custom `load_raw_data()` helper to inspect their dimensions, columns, and properties.

In [2]:
from src.data_loader import load_raw_data

train, test, stores, features = load_raw_data()

Loading raw data...


## 2. Train Dataset Schema & Summary
In this step, we examine the main historical dataset `train.csv`.
* **Target Variable**: `Weekly_Sales` represents the sales amount for a given store/department combination during that week.
* **Key Predictors**: `Store` identifier, `Dept` (department) identifier, `Date` of the week, and `IsHoliday` (whether it was a holiday week).
* **Observations**: There are 421,570 records in total, and no missing values are present in the core columns. However, the `Date` column is loaded as an `object` (string) type, which we will need to parse to a `datetime` format for time-series operations.

In [3]:
print("Shape:", train.shape)
print(train.info())
print("\nMissing Values:")
print(train.isnull().sum())
train.head()

Shape: (421570, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 421570 entries, 0 to 421569
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Store         421570 non-null  int64  
 1   Dept          421570 non-null  int64  
 2   Date          421570 non-null  object 
 3   Weekly_Sales  421570 non-null  float64
 4   IsHoliday     421570 non-null  bool   
dtypes: bool(1), float64(1), int64(2), object(1)
memory usage: 13.3+ MB
None

Missing Values:
Store           0
Dept            0
Date            0
Weekly_Sales    0
IsHoliday       0
dtype: int64


,Store,Dept,Date,Weekly_Sales,IsHoliday
0,1,1,2010-02-05,24924.50,False
1,1,1,2010-02-12,46039.49,True
2,1,1,2010-02-19,41595.55,False
3,1,1,2010-02-26,19403.54,False
4,1,1,2010-03-05,21827.90,False


## 3. Stores Metadata Schema & Summary
Next, we inspect the metadata for the physical stores (`stores.csv`).
* **Observations**: There are 45 unique stores. Each store is classified by `Type` (A, B, or C) and possesses a specific `Size` (represented as square footage). This metadata is crucial because store size and store type are likely to explain major baseline differences in weekly sales.

In [4]:
print("Shape:", stores.shape)
print(stores.info())
print("\nUnique Types:", stores['Type'].unique())
stores.head()

Shape: (45, 3)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45 entries, 0 to 44
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Store   45 non-null     int64 
 1   Type    45 non-null     object
 2   Size    45 non-null     int64 
dtypes: int64(2), object(1)
memory usage: 1.2+ KB
None

Unique Types: ['A' 'B' 'C']


,Store,Type,Size
0,1,A,151315
1,2,A,202307
2,3,B,37392
3,4,A,205863
4,5,B,34875


## 4. Features Dataset Schema & Summary
The `features.csv` dataset contains various environmental and macroeconomic factors.
* **Observations**: This dataset has 8,190 rows. Crucially, we observe significant missing values in the `MarkDown1-5` columns, as well as several missing entries in `CPI` and `Unemployment`.
* **Hypothesis**: The markdown variables likely contain missing values (`NaN`) when there was no active promotional campaign for that store-week. We will need to impute these to `0.0`. The missing values in `CPI` and `Unemployment` occur during the test set period, which we must fill using historical regional trends.

In [5]:
print("Shape:", features.shape)
print(features.info())
print("\nMissing Values:")
print(features.isnull().sum())
features.head()

Shape: (8190, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8190 entries, 0 to 8189
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Store         8190 non-null   int64  
 1   Date          8190 non-null   object 
 2   Temperature   8190 non-null   float64
 3   Fuel_Price    8190 non-null   float64
 4   MarkDown1     4032 non-null   float64
 5   MarkDown2     2921 non-null   float64
 6   MarkDown3     3613 non-null   float64
 7   MarkDown4     3464 non-null   float64
 8   MarkDown5     4050 non-null   float64
 9   CPI           7605 non-null   float64
 10  Unemployment  7605 non-null   float64
 11  IsHoliday     8190 non-null   bool   
dtypes: bool(1), float64(9), int64(1), object(1)
memory usage: 712.0+ KB
None

Missing Values:
Store              0
Date               0
Temperature        0
Fuel_Price         0
MarkDown1       4158
MarkDown2       5269
MarkDown3       4577
MarkDown4       4726
MarkDown5 

,Store,Date,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday
0,1,2010-02-05,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,False
1,1,2010-02-12,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,True
2,1,2010-02-19,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,False
3,1,2010-02-26,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,False
4,1,2010-03-05,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,False


## 5. Test Dataset Schema & Summary
Finally, we inspect the test dataset `test.csv`.
* **Observations**: The test set has 115,064 rows. As expected for an evaluation set, the target variable `Weekly_Sales` is absent. We must ensure that our cleaning pipeline outputs a test set that perfectly aligns with the schema of the training set (minus the target column) so that our forecasting models can run inference.

In [6]:
print("Shape:", test.shape)
print(test.info())
test.head()

Shape: (115064, 4)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 115064 entries, 0 to 115063
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   Store      115064 non-null  int64 
 1   Dept       115064 non-null  int64 
 2   Date       115064 non-null  object
 3   IsHoliday  115064 non-null  bool  
dtypes: bool(1), int64(2), object(1)
memory usage: 2.7+ MB
None


,Store,Dept,Date,IsHoliday
0,1,1,2012-11-02,False
1,1,1,2012-11-09,False
2,1,1,2012-11-16,False
3,1,1,2012-11-23,True
4,1,1,2012-11-30,False
